In [1]:
import pandas as pd 

# Normalization.py

In [2]:
import re
import pandas as pd
import jellyfish
import gc
from unidecode import unidecode

# 1. Suffixes and Stopwords
SUFFIX_MAP = {
    r'\bprivate limited\b': 'ltd', r'\bpvt ltd\b': 'ltd', r'\bpvt\.?\s*ltd\.?\b': 'ltd',
    r'\bcorporation\b': 'corp', r'\bincorporated\b': 'inc', r'\bstreet\b': 'st',
    r'\broad\b': 'rd', r'\bavenue\b': 'ave', r'\bboulevard\b': 'blvd'
}
COMPILED_SUFFIXES = [(re.compile(p), r) for p, r in SUFFIX_MAP.items()]

GEO_STOPWORDS = {
    "drive", "dr", "street", "st", "road", "rd", "avenue", "ave", "lane", "ln",
    "boulevard", "blvd", "no", "number", "plot", "flat", "shop", "floor", "building",
    "bldg", "near", "opp", "opposite", "behind", "city", "town", "pradesh", "state",
    "ltd", "inc", "corp", "co", "llc", "room", "suite", "ste", "apt", "apartment",
    "marg", "nagar", "vihar", "puram", "gali", "bhavan", "mahal", "complex", 
    "bagh", "chowk", "naka", "taluka", "zila", "mandal", "gram", "phase", "sector",
    "khand", "colony", "enclave", "extension", "ext", "cross", "main",
    "rue", "chemin", "place", "allee", "route", "impasse", "batiment", 
    "immeuble", "etage", "cedex", "bp", "boite", "postale"
}

def normalize_pipeline(df: pd.DataFrame, upper_threshold: int = 15000, lower_threshold: int = 50) -> pd.DataFrame:
    print(f"Normalizing {len(df)} rows via High-Speed Vectorization (No Multiprocessing)...")
    out = df.copy()
    
    names_raw = out['business_name'].tolist()
    addrs_raw = out['business_address'].tolist()
    countries_raw = out['country'].tolist()
    
    # 1. Ultra-fast list comprehensions (Bypasses Pandas .apply overhead)
    out['clean_name'] = [unidecode(str(x)).lower() if pd.notna(x) else "" for x in names_raw]
    out['clean_address'] = [unidecode(str(x)).lower() if pd.notna(x) else "" for x in addrs_raw]
    
    # 2. Vectorized Regex
    out['clean_name'] = out['clean_name'].str.replace(r'[^\w\s]', ' ', regex=True)
    out['clean_address'] = out['clean_address'].str.replace(r'[^\w\s]', ' ', regex=True)
    
    for pat, repl in COMPILED_SUFFIXES:
        out['clean_name'] = out['clean_name'].str.replace(pat, repl, regex=True)
        
    out['clean_name'] = out['clean_name'].str.replace(r'\s+', ' ', regex=True).str.strip()
    out['clean_address'] = out['clean_address'].str.replace(r'\s+', ' ', regex=True).str.strip()
    
    # 3. Geo Extraction Loop
    print("  -> Extracting Geo Layers...")
    geo_results = []
    for addr, country in zip(addrs_raw, countries_raw):
        c_str = str(country).lower().strip() if pd.notna(country) else ""
        r_addr = str(addr).lower().strip() if pd.notna(addr) else ""
        
        pin_match = re.search(r'\b\d{5,6}\b(?=[^\w]*$|\s*[a-z]{2,}\s*$)', r_addr)
        pin = pin_match.group(0) if pin_match else None
        
        chunks = [c.strip() for c in r_addr.split(',') if c.strip()]
        valid_chunks = []
        for chunk in chunks:
            chunk = unidecode(chunk)
            cleaned = re.sub(r'\b\d{5,6}\b', '', chunk)
            cleaned = re.sub(r'[^\w\s]', ' ', cleaned)
            cleaned = re.sub(r'\s+', ' ', cleaned).strip()
            if len(cleaned) > 1 and cleaned not in GEO_STOPWORDS and cleaned not in {c_str, "usa", "us", "india", "ind", "france"}:
                valid_chunks.append(cleaned.replace(" ", "_"))
                
        geo_results.append({
            "country": c_str, "pin": pin, 
            "chunk_1": valid_chunks[-1] if len(valid_chunks) >= 1 else None, 
            "chunk_2": valid_chunks[-2] if len(valid_chunks) >= 2 else None
        })
        
    geo_df = pd.DataFrame(geo_results, index=out.index)
    
    def assign_base_key(row):
        if row['pin']: return f"{row['country']}_{row['pin']}"
        elif row['chunk_1']: return f"{row['country']}_{row['chunk_1']}"
        return f"{row['country']}_unknown"
        
    out['geo_block_key'] = geo_df.apply(assign_base_key, axis=1)
    
    # 4. Threshold Applications
    print("  -> Applying Spatial Thresholds...")
    key_counts = out['geo_block_key'].value_counts()
    massive_keys = set(key_counts[key_counts > upper_threshold].index)
    
    out['geo_block_key'] = [
        f"{c}_{c2}_{c1}" if (k in massive_keys and not re.search(r'\d{5,6}$', k) and c2) else k 
        for k, c, c1, c2 in zip(out['geo_block_key'], geo_df['country'], geo_df['chunk_1'], geo_df['chunk_2'])
    ]
    
    post_counts = out['geo_block_key'].value_counts()
    mega_keys = set(post_counts[post_counts > upper_threshold].index)
    
    out['geo_block_key'] = [
        f"{k}_{name[0] if name else 'z'}" if (k in mega_keys and not re.search(r'\d{5,6}$', k)) else k
        for k, name in zip(out['geo_block_key'], out['clean_name'])
    ]
    
    final_counts = out['geo_block_key'].value_counts()
    valid_keys = set(final_counts[final_counts >= lower_threshold].index)
    out['geo_block_key'] = [k if k in valid_keys else f"{k.split('_')[0]}_unknown" for k in out['geo_block_key']]
    
    print("  -> Hashing Phonetics...")
    out['phonetic_hash'] = [jellyfish.metaphone(str(x)) if x else "" for x in out['clean_name']]
    out['search_document'] = out['clean_name'] + " " + out['clean_address']
    
    out = out.drop(columns=['business_name', 'business_address'])
    gc.collect()
    return out

# Blocking.py

In [3]:
import time
import gc
import numpy as np
import pandas as pd
import bm25s
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn


def _query_and_extract(
    retriever: bm25s.BM25, query_tokens, query_ids: np.ndarray, 
    target_ids: np.ndarray, score_col: str, top_k: int, n_threads: int
) -> pd.DataFrame:
    """Helper to query a BM25 index and extract non-empty matches with explicit threading."""
    # Explicitly pass n_threads to force the backend to utilize all cores
    results, scores = retriever.retrieve(query_tokens, k=top_k, n_threads=n_threads, show_progress=False)
    
    cand_indices = results.flatten()
    flat_scores = scores.flatten()
    
    query_indices = np.repeat(np.arange(len(query_ids)), top_k)
    valid_mask = cand_indices >= 0
    
    if not np.any(valid_mask):
        return pd.DataFrame(columns=['entity_A', 'entity_B', score_col])
        
    return pd.DataFrame({
        'entity_A': query_ids[query_indices[valid_mask]],
        'entity_B': target_ids[cand_indices[valid_mask]],
        score_col: flat_scores[valid_mask]
    })


def run_country_global_scan(
    s1_sub: pd.DataFrame, s2_sub: pd.DataFrame, s3_sub: pd.DataFrame,
    country_str: str, modality: str, top_k: int = 7, chunk_size: int = 50000, n_threads: int = 4
) -> pd.DataFrame:
    """Hybrid Architecture: Index once, but tokenize and query in safe CPU chunks."""
    print(f"\n[{country_str.upper()} | {modality.upper()}] S1: {len(s1_sub)} | S2: {len(s2_sub)} | S3: {len(s3_sub)}")
    t0 = time.time()
    score_col = f'bm25_{modality}_score'
    
    # 1. Text Selection
    if modality == 'text':
        s1_text = s1_sub['search_document'].fillna("").tolist()
        s2_text = s2_sub['search_document'].fillna("").tolist()
        s3_text = s3_sub['search_document'].fillna("").tolist()
    else:
        s1_text = s1_sub['phonetic_hash'].fillna("").tolist()
        s2_text = s2_sub['phonetic_hash'].fillna("").tolist()
        s3_text = s3_sub['phonetic_hash'].fillna("").tolist()
        
    pairs = []
    
    # 2. Build Target Indices Once
    retriever_s2, retriever_s3 = None, None
    
    if len(s2_sub) > 0:
        print("  -> Pre-tokenizing and Building S2 Index...")
        s2_tokens_for_index = bm25s.tokenize(s2_text)
        retriever_s2 = bm25s.BM25()
        retriever_s2.index(s2_tokens_for_index)
        del s2_tokens_for_index  # Free RAM immediately
        
    if len(s3_sub) > 0:
        print("  -> Pre-tokenizing and Building S3 Index...")
        s3_tokens_for_index = bm25s.tokenize(s3_text)
        retriever_s3 = bm25s.BM25()
        retriever_s3.index(s3_tokens_for_index)
        del s3_tokens_for_index  # Free RAM immediately
        
    s1_ids = s1_sub['entity_id'].values
    s2_ids = s2_sub['entity_id'].values
    s3_ids = s3_sub['entity_id'].values

    # 3. CPU-Friendly Chunked Querying
    if len(s1_sub) > 0 and (retriever_s2 is not None or retriever_s3 is not None):
        print(f"  -> Querying S1 across targets in {chunk_size} row batches...")
        for i in range(0, len(s1_ids), chunk_size):
            chunk_end = min(i + chunk_size, len(s1_ids))
            
            # THE FIX: Tokenize the text chunk directly. Bypass JaggedArray slicing entirely.
            chunk_tokens = bm25s.tokenize(s1_text[i:chunk_end])
            
            if retriever_s2 is not None:
                pairs.append(_query_and_extract(retriever_s2, chunk_tokens, s1_ids[i:chunk_end], s2_ids, score_col, top_k, n_threads))
            if retriever_s3 is not None:
                pairs.append(_query_and_extract(retriever_s3, chunk_tokens, s1_ids[i:chunk_end], s3_ids, score_col, top_k, n_threads))
            
            print(f"     ...Processed S1 queries: {chunk_end}/{len(s1_ids)}")

    if len(s2_sub) > 0 and retriever_s3 is not None:
        print(f"  -> Querying S2 against S3 in {chunk_size} row batches...")
        for i in range(0, len(s2_ids), chunk_size):
            chunk_end = min(i + chunk_size, len(s2_ids))
            
            # Tokenize S2 directly inside the loop as well
            chunk_tokens = bm25s.tokenize(s2_text[i:chunk_end])
            
            pairs.append(_query_and_extract(retriever_s3, chunk_tokens, s2_ids[i:chunk_end], s3_ids, score_col, top_k, n_threads))
            print(f"     ...Processed S2 queries: {chunk_end}/{len(s2_ids)}")

    # Extreme RAM wipe
    del retriever_s2, retriever_s3
    del s1_text, s2_text, s3_text
    gc.collect()
    
    merged = pd.concat(pairs, ignore_index=True) if pairs else pd.DataFrame(columns=['entity_A', 'entity_B', score_col])
    print(f"  -> Generated {len(merged)} candidate pairs in {time.time() - t0:.2f}s.")
    return merged

def generate_global_candidate_pool(
    s1: pd.DataFrame, s2: pd.DataFrame, s3: pd.DataFrame, 
    top_k: int = 7, chunk_size: int = 50000, n_threads: int = 4
) -> pd.DataFrame:
    """Executes global candidate extraction with hybrid batch chunking."""
    print("=" * 40)
    print("STAGES 1-3: GLOBAL BM25 CANDIDATE EXTRACTION")
    print("=" * 40)
    
    countries = set(s1['country'].str.lower().dropna()) | \
                set(s2['country'].str.lower().dropna()) | \
                set(s3['country'].str.lower().dropna())
                
    country_candidates = []
    for country in sorted(countries):
        s1_sub = s1[s1['country'].str.lower() == country]
        s2_sub = s2[s2['country'].str.lower() == country]
        s3_sub = s3[s3['country'].str.lower() == country]
        
        if s1_sub.empty and s2_sub.empty and s3_sub.empty:
            continue
            
        text_df = run_country_global_scan(s1_sub, s2_sub, s3_sub, country, 'text', top_k, chunk_size, n_threads)
        phonetic_df = run_country_global_scan(s1_sub, s2_sub, s3_sub, country, 'phonetic', top_k, chunk_size, n_threads)
        
        print(f"  -> Merging text & phonetic candidates for {country.upper()}...")
        country_pairs = pd.merge(text_df, phonetic_df, on=['entity_A', 'entity_B'], how='outer').fillna(0.0)
        country_candidates.append(country_pairs)
        
    if not country_candidates:
        return pd.DataFrame(columns=['entity_A', 'entity_B', 'bm25_text_score', 'bm25_phonetic_score'])
        
    master_global = pd.concat(country_candidates, ignore_index=True)
    master_global = master_global.drop_duplicates(subset=['entity_A', 'entity_B'])
    
    print(f"\nTotal Unified Global BM25 Pairs: {len(master_global)}")
    return master_global


def extract_top_k_pairs(s1_ids: np.ndarray, s2_ids: np.ndarray, matches_matrix, score_col: str) -> pd.DataFrame:
    """Rapid sparse matrix extraction."""
    nonzeros = matches_matrix.nonzero()
    return pd.DataFrame({
        'entity_A': s1_ids[nonzeros[0]],
        'entity_B': s2_ids[nonzeros[1]],
        score_col: matches_matrix.data
    })


def execute_local_geo_index(s1_df: pd.DataFrame, s2_df: pd.DataFrame, top_k: int = 7) -> pd.DataFrame:
    """Executes fuzzy Character N-Grams strictly within valid local geographic buckets."""
    valid_blocks = {b for b in (set(s1_df['geo_block_key'].unique()) & set(s2_df['geo_block_key'].unique())) if not b.endswith('_unknown')}
    
    if not valid_blocks:
        return pd.DataFrame(columns=['entity_A', 'entity_B', 'geo_ngram_score'])
        
    print(f"  -> Sweeping {len(valid_blocks)} shared geographic blocks...")
    all_geo_pairs = []
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 4), min_df=1)
    
    for block in valid_blocks:
        s1_sub = s1_df[s1_df['geo_block_key'] == block]
        s2_sub = s2_df[s2_df['geo_block_key'] == block]
        
        s2_matrix = vectorizer.fit_transform(s2_sub['search_document'].fillna(""))
        s1_matrix = vectorizer.transform(s1_sub['search_document'].fillna(""))
        matches_matrix = sp_matmul_topn(s1_matrix, s2_matrix.T.tocsr(), top_n=min(top_k, len(s2_sub)))
        
        pairs = extract_top_k_pairs(s1_sub['entity_id'].values, s2_sub['entity_id'].values, matches_matrix, 'geo_ngram_score')
        all_geo_pairs.append(pairs)
        
    return pd.concat(all_geo_pairs, ignore_index=True) if all_geo_pairs else pd.DataFrame(columns=['entity_A', 'entity_B', 'geo_ngram_score'])


def run_multi_source_geo_blocking(s1: pd.DataFrame, s2: pd.DataFrame, s3: pd.DataFrame, top_k: int = 7) -> pd.DataFrame:
    print("\n" + "=" * 40)
    print("STAGE 4: INDEX 3 (LOCAL GEO BLOCKING)")
    print("=" * 40)
    
    geo_12 = execute_local_geo_index(s1, s2, top_k)
    geo_13 = execute_local_geo_index(s1, s3, top_k)
    geo_23 = execute_local_geo_index(s2, s3, top_k)
    
    master_geo = pd.concat([geo_12, geo_13, geo_23], ignore_index=True).drop_duplicates(subset=['entity_A', 'entity_B'])
    print(f"\nTotal Local Geo Pairs Generated: {len(master_geo)}")
    return master_geo

## Loading data 

In [4]:
s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t')
print("s1 loaded")
s2 = pd.read_csv("../student_resource/dataset/train/train_source2.tsv", sep='\t')
print("s2 loaded")
s3 = pd.read_csv("../student_resource/dataset/train/train_source3.tsv", sep='\t')
print("s3 loaded")
gt = pd.read_csv("../student_resource/dataset/train/train_ground_truth.tsv", sep='\t')
print("gt loaded")

s1 loaded
s2 loaded
s3 loaded
gt loaded


--- 
# Normalization

In [5]:
s1_normalized = normalize_pipeline(s1)
print("S1 normalized")
s2_normalized = normalize_pipeline(s2)
print("S2 normalized")
s3_normalized = normalize_pipeline(s3)
print("S3 normalized")

Normalizing 2206821 rows via High-Speed Vectorization (No Multiprocessing)...
  -> Extracting Geo Layers...
  -> Applying Spatial Thresholds...
  -> Hashing Phonetics...
S1 normalized
Normalizing 5034616 rows via High-Speed Vectorization (No Multiprocessing)...
  -> Extracting Geo Layers...
  -> Applying Spatial Thresholds...
  -> Hashing Phonetics...
S2 normalized
Normalizing 5285603 rows via High-Speed Vectorization (No Multiprocessing)...
  -> Extracting Geo Layers...
  -> Applying Spatial Thresholds...
  -> Hashing Phonetics...
S3 normalized


## Standardization and Blocking

In [ ]:
import polars as pl

# 1. Global Scan 
global_candidates_df = generate_global_candidate_pool(
    s1_normalized, 
    s2_normalized, 
    s3_normalized, 
    top_k=7
)

# 2. Local Geo Scan
geo_candidates_df = run_multi_source_geo_blocking(
    s1_normalized, 
    s2_normalized, 
    s3_normalized, 
    top_k=7
)

# 3. Polars Outer Merge (Bypasses Pandas memory explosion)
print("\nConverting to Polars for Rust-level outer merge...")
pl_global = pl.from_pandas(global_candidates_df)
pl_geo = pl.from_pandas(geo_candidates_df)

print("\nMerging Local Geo matches into Master Candidate Pool...")
pl_master = pl_global.join(
    pl_geo, 
    on=['entity_A', 'entity_B'], 
    how='full', 
    coalesce=True
).fill_null(0.0)

final_master_pool = pl_master.to_pandas()
print(f"Final Candidate Pool Size: {len(final_master_pool)}")

STAGES 1-3: GLOBAL BM25 CANDIDATE EXTRACTION

[INDIA | TEXT] S1: 883188 | S2: 2017799 | S3: 2115547
  -> Pre-tokenizing and Building S2 Index...


Split strings:   0%|          | 0/2017799 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/2017799 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/2017799 [00:00<?, ?it/s]

  -> Pre-tokenizing and Building S3 Index...


...

Split strings:   0%|          | 0/50000 [00:00<?, ?it/s]

     ...Processed S1 queries: 150000/883188


Split strings:   0%|          | 0/50000 [00:00<?, ?it/s]

     ...Processed S1 queries: 200000/883188


Split strings:   0%|          | 0/50000 [00:00<?, ?it/s]